# Project 1

by Akiva Nugent and Ryan Buck

### Setup

In [ ]:
import gradio as gr
from random import randint
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
client = OpenAI()

### Our Prompt

In [ ]:
scenarios = [
    {
        "Symptoms": "No power\nCorrosion on the port",
        "Background": "Phone near pool, though not directly submerged in the water",
        "Likely Cause": "Liquid damage leading to a corroded logic board",
        "Repairable?": "Unlikely (no)",
    },
    {
        "Symptoms": "No power\nCracked screen",
        "Background": "Recent hard drop",
        "Likely Cause": "Physical trauma damage",
        "Repairable?": "Often repairable",
    },
    {
        "Symptoms": "No power",
        "Background": "Slow decline over a number of weeks\nOld phone",
        "Likely Cause": "Battery end-of-life",
        "Repairable?": "Usually repairable",
    },
    {
        "Symptoms": "No power\nBattery flickers on then dies",
        "Background": "No specific history",
        "Likely Cause": "Software/firmware issue",
        "Repairable?": "Usually repairable",
    }
]

persona_prompt = """
    You are a hostile customer. You should enter the conversation in an angry state. 

    Occasionally, you should ignore some details, in line with the personality of an angry customer.

    Do not make up rudeness when it is not there, but you should be quick to escalate when the trainee is rude or unhelpful. 

    The trainee's role is to make you feel heard and understood. 
"""

def random_scenario_index():
    return randint(1, len(scenarios) - 1)

def build_context_prompt(scenario):
    return f"""
        You are a customer with a broken mobile device. The device will not turn on, but you do not know the underlying cause. 

        As a training bot, you know the scenario and acceptable resolutions. In the role of the customer, you do not know the possible outcomes.

        Symptoms:
        {scenario["Symptoms"]}

        Background/History:
        {scenario["Background"]}
    """

def build_system_prompt(scenario_index):
    scenario = scenarios[scenario_index]
    context_prompt = build_context_prompt(scenario)

    system_prompt = f"""
        You are a customer-service-agent  training partner- please keep going until the user's training is completely resolved. 

        Context:
        {context_prompt}

        Personality
        {persona_prompt}
    """

    return system_prompt

assess_prompt = """
    Briefly assess the trainee's message, think about how you should respond, determine the customer's current state and determine if the trainee has committed a violation.

    At the end of your message write a JSON object similar to this example:
    ```
    {
        "customer_state": "frustrated",
        "violation_counter": 0
    }
    ```

    Do NOT write anything after the JSON!
"""

respond_prompt = """
    Now respond to the trainee.
"""

grade_prompt = """
    Grade... WIP PLACEHOLDER
"""


### Chatbot

In [ ]:
class Bot:
    def __init__(self):
        self.scenario_index = random_scenario_index()
        self.system_prompt = build_system_prompt(self.scenario_index)

    def _create_sys_msg(self, msg):
        return {
            "role": "system",
            "content": msg
        }

    # def grade(self, history):
    #     if history[-1].content:
    #         pass

    def respond(self, history):
        system_msg = self._create_sys_msg(self.system_prompt)

        # Assess Trainee

        assess_msg = self._create_sys_msg(assess_prompt)
        assess_history = [system_msg] + history + [assess_msg]

        assess_response = client.chat.completions.create(
            model="gpt-4o",
            messages=assess_history,
        )

        # Respond to Trainee

        respond_msg = self._create_sys_msg(respond_prompt)
        respond_history = assess_history + [assess_response.choices[0].message, respond_msg]

        stream = client.chat.completions.create(
            model="gpt-4o",
            messages=respond_history,
            stream=True
        )

        history = history + [{"role": "assistant", "content": ""}]
        for chunk in stream:
            if chunk.choices and chunk.choices[0].delta.content:
                history[-1]["content"] += chunk.choices[0].delta.content
                yield history


### Chatbot & UI

In [ ]:
def user_submit(message, history):
    # Add the user's message to the history, and clear the textbox
    history = history + [{"role": "user", "content": message}]
    return history, ""

def start_message():
    return [{
        "role": "assistant",
        "content": "Hello!"
    }]

gr.close_all()

bot = Bot()

with gr.Blocks(title="Chat", fill_width=True) as demo:
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 📄")
        with gr.Column(scale=4):
            gr.Markdown("### Chat")
            gr.Markdown("*Powered by Champlain College*")

    gr.Markdown("---")

    with gr.Sidebar():
        reset_btn = gr.Button("🔄 Reset Conversation")

    chatbot = gr.Chatbot(
        value=start_message(),
        label="Chat",
        height=400,
    )
    msg = gr.Textbox(
        placeholder="Enter your message...",
        label="Your message",
    )

    msg.submit(user_submit, inputs=[msg, chatbot], outputs=[chatbot, msg]).then(
        bot.respond, inputs=[chatbot], outputs=[chatbot]
    )

    reset_btn.click(start_message, outputs=[chatbot])

demo.launch()
